## scPRINT ingestion

Environment setup:

```bash
uv venv .scprint --python 3.11
source .scprint/bin/activate

uv pip install scprint ipykernel "napistu-torch>=0.3.8"
python -m ipykernel install --user --name=scPRINT

# Initialize lamin database for gene annotations (optional but recommended)
lamin init --storage data/lamin_db --name scPRINT_lamin --modules bionty
```

In [1]:
import logging
import os

# Set up logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

from napistu.genomics.scverse_loading import DatasetsConfig

from napistu_torch.load.foundation_model_etl import (
    populate_lamin_db,
    process_scprint,
)
from napistu_torch.load.foundation_models import FoundationModel
from napistu_torch.load.constants import (
    FM_DEFS,
    FOUNDATION_MODEL_NAMES,
    SCPRINT_DEFS,
)
import numpy as np

logger = logging.getLogger(__name__)

INFO:numexpr.utils:NumExpr defaulting to 16 threads.


In [2]:
# Configuration
DATA_DIR = "data"
OUTPUT_DIR = "output"
# Raw config dictionary
DATASETS_CONFIG = {
    "efthymiou2025": {
        "uri": "https://cellxgene.cziscience.com/collections/6b701826-37bb-4356-9792-ff41fc4c3161",
        "path": os.path.expanduser("~/Desktop/DATA/genomics/efthymiou.h5ad")
    }
}

SCPRINT_MODEL_PATH = os.path.join(DATA_DIR, FOUNDATION_MODEL_NAMES.SCPRINT)
os.makedirs(SCPRINT_MODEL_PATH, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Validated config
datasets_config = DatasetsConfig(DATASETS_CONFIG)


In [3]:
SCPRINT_VERSION_KEYS = list(SCPRINT_DEFS.VERSIONS.__dict__.keys())

populate_lamin_db()

for version in SCPRINT_VERSION_KEYS:
    process_scprint(version, OUTPUT_DIR, SCPRINT_MODEL_PATH, datasets_config)

→ connected lamindb: anonymous/scPRINT_lamin
No module named 'triton'
FlashAttention is not installed, not using it..
RuntimeError caught: scPrint is not attached to a `Trainer`.
RuntimeError caught: scPrint is not attached to a `Trainer`.
RuntimeError caught: scPrint is not attached to a `Trainer`.


In [4]:
# Load results for a specific version (using MEDIUM as an example)
medium_version_id = SCPRINT_DEFS.VERSIONS.MEDIUM
file_prefix = f"{FOUNDATION_MODEL_NAMES.SCPRINT}_{medium_version_id}"
model = FoundationModel.load(OUTPUT_DIR, file_prefix)

GENES_OF_INTEREST = model.gene_annotations[FM_DEFS.VOCAB_NAME].sample(20000).tolist()
GENE_MASK = [x in GENES_OF_INTEREST for x in model.ordered_vocabulary]

# Compute attention on demand using FoundationModelWeights method
# This handles multi-head attention properly
layer_attn = model.weights.compute_attention_from_weights(
    layer_idx=3,
    n_heads=model.n_heads,
    gene_mask=np.array(GENE_MASK)
)

model.dataset_gene_embeddings

DatasetGeneEmbeddings(n_datasets=1, total_embeddings=19, datasets=['efthymiou2025'])